Load Wine dataset + train/test split + MinMax scaling

In [1]:
import numpy as np
import pandas as pd

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

# Reproducibility
np.random.seed(42)

# Load dataset (3 classes)
wine = load_wine()
X = wine.data
y = wine.target

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Number of classes:", len(np.unique(y)))

# Train/test split (similar to lecture style)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=42, stratify=y
)

# Scale features (Lecture 2 tabular example uses MinMaxScaler)
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print("Train shapes:", X_train.shape, y_train.shape)
print("Test shapes:", X_test.shape, y_test.shape)

X shape: (178, 13)
y shape: (178,)
Number of classes: 3
Train shapes: (119, 13) (119,)
Test shapes: (59, 13) (59,)


Baseline MLP for Wine classification (ReLU + Softmax, Adam)

In [2]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

tf.random.set_seed(42)

# Baseline model (similar to Lecture 2 Iris example)
model_base = Sequential([
    Dense(10, activation="relu", kernel_initializer="he_normal", input_shape=(X_train.shape[1],)),
    Dense(8, activation="relu", kernel_initializer="he_normal"),
    Dense(3, activation="softmax")
])

model_base.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_base = model_base.fit(
    X_train, y_train,
    epochs=200,
    batch_size=32,
    validation_data=(X_test, y_test),
    verbose=0
)

loss_base, acc_base = model_base.evaluate(X_test, y_test, verbose=0)
print("Baseline Test loss:", loss_base)
print("Baseline Test accuracy:", acc_base)

c:\Users\hasan\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Baseline Test loss: 0.12558847665786743
Baseline Test accuracy: 0.9491525292396545


Improved MLP for Wine: larger layers + Dropout (better generalization)

In [3]:
from tensorflow.keras.layers import Dropout

tf.random.set_seed(42)

model_improved = Sequential([
    Dense(32, activation="relu", kernel_initializer="he_normal", input_shape=(X_train.shape[1],)),
    Dropout(0.2),
    Dense(16, activation="relu", kernel_initializer="he_normal"),
    Dropout(0.2),
    Dense(3, activation="softmax")
])

model_improved.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_improved = model_improved.fit(
    X_train, y_train,
    epochs=300,
    batch_size=16,
    validation_data=(X_test, y_test),
    verbose=0
)

loss_imp, acc_imp = model_improved.evaluate(X_test, y_test, verbose=0)
print("Improved Test loss:", loss_imp)
print("Improved Test accuracy:", acc_imp)

Improved Test loss: 0.042848482728004456
Improved Test accuracy: 0.9830508232116699


Comparison table (preprocessing + hyperparameters + performance)

In [4]:
results_22 = pd.DataFrame([
    {
        "Model": "Baseline MLP",
        "Preprocessing": "MinMaxScaler",
        "Architecture": "13 -> 10(ReLU) -> 8(ReLU) -> 3(Softmax)",
        "Optimizer": "Adam",
        "Loss": "sparse_categorical_crossentropy",
        "Epochs": 200,
        "Batch size": 32,
        "Dropout": "No",
        "Test loss": loss_base,
        "Test accuracy": acc_base
    },
    {
        "Model": "Improved MLP",
        "Preprocessing": "MinMaxScaler",
        "Architecture": "13 -> 32(ReLU) -> Dropout(0.2) -> 16(ReLU) -> Dropout(0.2) -> 3(Softmax)",
        "Optimizer": "Adam",
        "Loss": "sparse_categorical_crossentropy",
        "Epochs": 300,
        "Batch size": 16,
        "Dropout": "Yes (0.2, 0.2)",
        "Test loss": loss_imp,
        "Test accuracy": acc_imp
    }
])

results_22

,Model,Preprocessing,Architecture,Optimizer,Loss,Epochs,Batch size,Dropout,Test loss,Test accuracy
0,Baseline MLP,MinMaxScaler,13 -> 10(ReLU) -> 8(ReLU) -> 3(Softmax),Adam,sparse_categorical_crossentropy,200,32,No,0.125588,0.949153
1,Improved MLP,MinMaxScaler,13 -> 32(ReLU) -> Dropout(0.2) -> 16(ReLU) -> ...,Adam,sparse_categorical_crossentropy,300,16,"Yes (0.2, 0.2)",0.042848,0.983051
